In [10]:
import numpy as np
import librosa
import IPython.display as ipd

In [17]:
def make_stereo(audio_paths, mix_matrix, sr=None):
  audios = []
  rates = []

  for path in audio_paths:
    audio, rate = librosa.load(path, sr=sr, mono=True)
    audios.append(audio)
    rates.append(rate)
  
  # Check sample rates
  if sr is None and not all(r == rates[0] for r in rates):
    raise ValueError("Sample rates must match. Set sr to fixed value if needed.")

  # Pad or trim to same length
  max_len = max(len(a) for a in audios)
  audios_equal = [np.pad(a, (0, max_len - len(a)), 'constant') if len(a) < max_len else a[:max_len] for a in audios]
  
  stacked_audio = np.stack(audios_equal, axis=-1)  # shape (num_samples, num_channels)

  # Check mix_matrix shape
  num_channels = len(audio_paths)
  if mix_matrix.shape[1] != num_channels:
    raise ValueError(f"mix_matrix should have {num_channels} columns.")

  # Apply mixing: mixes = stacked_audio @ mix_matrix.T
  mixes = np.dot(stacked_audio, mix_matrix.T)  # shape (num_samples, num_mixes)

  return mixes.T, rates[0] if sr is None else sr

In [15]:
audio_paths = [
  "./Dataset/ICA/spc1.wav",
  "./Dataset/ICA/spc2.wav",
  "./Dataset/ICA/spc3.wav"
]

mix_matrix = np.array([
  [1, 0, 0],      # only first audio
  [0, 1, 1],      # second + third audio
  [0.5, 0.5, 0]   # average of first and second
])

audio, sr = make_stereo(audio_paths, mix_matrix, 8000)

In [16]:
ipd.Audio(data=audio, rate=8000)